In [ ]:
from pathlib import Path
import sys

# Make this package's scripts/ importable (exp_info, regions, data_utils, ...).
PROJECT_ROOT = next(
    (path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (path / "scripts" / "exp_info.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start this notebook from inside the polar_analysis package.")

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

In [ ]:
# Parameters: directories and settings for this run. Edit them for another
# machine or experiment, or override with papermill (-p NAME VALUE).
REGION    = "Arctic"  # Arctic, Greenland, Antarctic, Atlantic (config/regions.json)
CLIMO_DIR = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/v3LE_paper/data/climo"  # input climatologies
DIAG_DIR  = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/v3LE_paper/diag_data"  # output diagnostic data (<REGION>/ is added)
FIG_DIR   = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/v3LE_paper/figures/mean_bias"  # output figures (<REGION>/ is added)

In [2]:
from __future__ import annotations

import os
import glob
import re
import json
import time
import datetime
from typing import Dict, Tuple, Optional
from global_land_mask import globe

import xarray as xr

import unicodedata
import pprint 
import shutil

from typing import Optional

try:
    # xCDAT for CF bounds utilities and lon handling
    from xcdat import open_dataset as xcdat_open
    _HAVE_XCDAT = True
except Exception:
    xcdat_open = xr.open_dataset
    _HAVE_XCDAT = False

try:
    import xskillscore as xs
    _HAVE_XS = True
except Exception:
    _HAVE_XS = False

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as matcolors
import matplotlib.path as matpath
import matplotlib.ticker as mticker
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from collections.abc import Mapping

from matplotlib.cm import ScalarMappable
from matplotlib.offsetbox import AnchoredText

# Local imports
from exp_info import RUN_CATALOG, REGION_CATALOG, VARIABLE_CATALOG


In [3]:
class ExpDataReader:
    def __init__(
        self,
        data_dir: str,
        regnam: str,
        dtype: str,
        *,
        run_catalog: Optional[Dict[str, Dict[str, str]]] = None,
        region_catalog: Optional[Dict[str, Tuple[Tuple[float, float], Tuple[float, float]]]] = None,
        sfc_mask_ref: str = "MODIS.observation",
        sfc_mask_var: str = "TS",
        sfc_mask_kind: str = "land",
        debug_key: Optional[str] = None,
        verbose: bool = False,
    ):
        self.data_dir = os.path.abspath(data_dir)
        self.regnam = regnam
        self.dtype = dtype
        self.sfc_mask_ref = sfc_mask_ref
        
        # copy to avoid accidental mutation of module-level defaults
        self.run_catalog = dict(RUN_CATALOG) if run_catalog is None else dict(run_catalog)
        self.region_catalog = dict(REGION_CATALOG) if region_catalog is None else dict(region_catalog)

        if self.sfc_mask_ref not in self.run_catalog:
            raise KeyError(f"sfc_mask_ref='{self.sfc_mask_ref}' not found in run_catalog")

        self.sfc_mask_var = sfc_mask_var
        self.sfc_mask_kind = sfc_mask_kind
        self.debug_key = debug_key
        self.verbose = verbose

        # region bounds (store both slices and raw bounds)
        (lat_min, lat_max), (lon_min, lon_max) = self._region_bounds(self.regnam)
        lon_min, lon_max = self._normalize_region_lon_bounds(lon_min, lon_max)
        self.lat_slice = slice(lat_min, lat_max)
        self.lon_slice = slice(lon_min, lon_max)
        self.lat_bounds = (lat_min, lat_max)
        self.lon_bounds = (lon_min, lon_max)

        self.exp_dict: Dict[str, xr.Dataset] = {}
        self.sfc_mask: Optional[xr.DataArray] = None

    def _meta_field(self, meta, field: str):
        """Return run metadata field from object or dict."""
        if hasattr(meta, field):
            return getattr(meta, field)
        if isinstance(meta, dict) and field in meta:
            return meta[field]
        raise KeyError(f"Run meta missing '{field}': {field} not found in {meta!r}")

    def gen_region_mask(self, data_dir, exp, dtype, period, var, regnam, debug=False):
        """
        Build a region-aware mask aligned to the reference grid.
        - base mask: finite data in `var` (1 where finite, 0 where NaN)
        - optional land/ocean filter via global_land_mask (if self.sfc_mask_kind in {'land','ocean'})
        - cropped to the region `regnam`
        Returns: xr.DataArray int8 with dims ('lat','lon') named 'region_mask'
        """
        # --- open & normalize lon ---
        refpath = os.path.join(data_dir, f"{exp}.{dtype}.{period}.nc")
        refds = xcdat_open(refpath, decode_times=True)

        lon_name = "lon" if "lon" in refds.coords else ("longitude" if "longitude" in refds.coords else None)
        if lon_name is None:
            raise ValueError("No longitude coordinate found ('lon' or 'longitude').")

        lons = refds[lon_name]
        # wrap 0–360 → −180..180 and sort
        if float(lons.min()) > -1.0 or float(lons.max()) > 180.0:
            new_lons = ((lons + 180) % 360) - 180
            refds = refds.assign_coords({lon_name: new_lons})
        refds = refds.sortby(lon_name)
        if lon_name == "longitude":
            refds = refds.rename({"longitude": "lon"})

        # ensure bounds (optional)
        refds = refds.bounds.add_missing_bounds()

        rlats = refds["lat"].values
        rlons = refds["lon"].values

        # --- get a 2D slice of var for base valid-data mask ---
        src = refds[var]
        drop_dims = [d for d in src.dims if d not in ("lat", "lon")]
        src2d = src.isel({d: 0 for d in drop_dims}).squeeze() if drop_dims else src.squeeze()

        # base mask: 1 where data finite, else 0
        base = xr.where(xr.apply_ufunc(np.isfinite, src2d), 1, 0).astype(np.int8).rename("region_mask")

        if debug:
            tot = int(base.size); val = int((base > 0).sum())
            print(f"[mask-base] finite cells={val}/{tot} ({val/tot:.1%})")

        # --- optional land/ocean filter ---
        kind = getattr(self, "sfc_mask_kind", None)
        if kind in {"land", "ocean"}:
            try:
                from global_land_mask import globe
            except Exception as e:
                raise ImportError("Please `pip install global-land-mask` to use land/ocean masking.") from e

            lon_grid, lat_grid = np.meshgrid(rlons, rlats)
            reg_bool = globe.is_land(lat_grid, lon_grid) if kind == "land" else globe.is_ocean(lat_grid, lon_grid)
            reg_mask = xr.DataArray(reg_bool, coords={"lat": rlats, "lon": rlons}, dims=("lat", "lon"))
            base = xr.where(reg_mask, base, 0).astype(np.int8)

            if debug:
                val2 = int((base > 0).sum()); tot2 = int(base.size)
                print(f"[mask-{kind}] kept cells={val2}/{tot2} ({val2/tot2:.1%})")

        # --- ensure latitude ascending for consistent slicing ---
        if np.diff(base["lat"].values).mean() < 0:
            base = base.sortby("lat")

        # --- crop to region ---
        (lat_min, lat_max), (lon_min, lon_max) = self._region_bounds(regnam)
        lon_min, lon_max = self._normalize_region_lon_bounds(lon_min, lon_max)

        out = base.sel(lat=slice(lat_min, lat_max))

        if lon_min <= lon_max:
            out = out.sel(lon=slice(lon_min, lon_max))
        else:
            left  = out.sel(lon=slice(lon_min, 180))
            right = out.sel(lon=slice(-180, lon_max))
            out = xr.concat([left, right], dim="lon").sortby("lon")

        out = out.astype(np.int8).rename("region_mask")
        out.attrs.update({
            "region": regnam,
            "lon_bounds": (lon_min, lon_max),
            "lat_bounds": (lat_min, lat_max),
            "kind": getattr(self, "sfc_mask_kind", "any"),
        })

        if debug:
            totR = int(out.size); valR = int((out > 0).sum())
            print(f"[crop-{regnam}] kept cells={valR}/{totR} ({valR/totR:.1%}); "
                  f"extent=[{lon_min:.1f},{lon_max:.1f}]×[{lat_min:.1f},{lat_max:.1f}]")

        return out

    def _region_bounds(self, regnam: str):
        """Return ((lat_min, lat_max), (lon_min, lon_max)) from Region object/dict/tuple."""
        try:
            region = self.region_catalog[regnam]
        except KeyError:
            raise ValueError(f"Region '{regnam}' not found. Available: {list(self.region_catalog)}")

        # Region dataclass / object
        if hasattr(region, "lat_range") and hasattr(region, "lon_range"):
            (lat_min, lat_max) = region.lat_range
            (lon_min, lon_max) = region.lon_range
            return (float(lat_min), float(lat_max)), (float(lon_min), float(lon_max))

        # Dict style
        if isinstance(region, dict) and "lat_range" in region and "lon_range" in region:
            (lat_min, lat_max) = region["lat_range"]
            (lon_min, lon_max) = region["lon_range"]
            return (float(lat_min), float(lat_max)), (float(lon_min), float(lon_max))

        # Tuple/list style
        try:
            (lat_min, lat_max), (lon_min, lon_max) = region
            return (float(lat_min), float(lat_max)), (float(lon_min), float(lon_max))
        except Exception:
            raise TypeError(
                "REGION_CATALOG entry must be a Region with lat_range/lon_range, "
                "a dict with those keys, or ((lat_min,lat_max),(lon_min,lon_max))."
            )
        
    # --- longitude handling ---
    @staticmethod
    def _is_0_360(lons) -> bool:
        """
        Detect if longitude grid is in [0, 360).
        Works even for shifted grids like 0.5–359.5.
        """
        a = np.asarray(lons)
        minlon, maxlon = np.nanmin(a), np.nanmax(a)
        if minlon >= 0.0 and maxlon <= 360.0:
            return True
        if (0.0 <= minlon < 1.0) and (359.0 < maxlon <= 360.0):
            return True
        return False

    @staticmethod
    def _wrap_to_180(lon_vals):
        lon_vals = (lon_vals + 180.0) % 360.0 - 180.0
        return lon_vals

    def _normalize_longitude(self, ds: xr.Dataset) -> xr.Dataset:
        """Ensure ds.lon is in [-180, 180] and sorted ascending."""
        lon_name = "lon" if "lon" in ds.coords else ("longitude" if "longitude" in ds.coords else None)
        if lon_name is None:
            return ds  # nothing to do

        lons = ds[lon_name].values
        if self._is_0_360(lons):
            lons = self._wrap_to_180(lons)
            ds = ds.assign_coords({lon_name: lons})
        # always sort by lon
        ds = ds.sortby(lon_name)

        # For consistency rename to 'lon' if it's 'longitude'
        if lon_name == "longitude":
            ds = ds.rename({lon_name: "lon"})
        return ds

    @staticmethod
    def _normalize_region_lon_bounds(lon_min: float, lon_max: float):
        """Ensure region bounds use the [-180, 180] convention."""
        if -180.0 <= lon_min <= 180.0 and -180.0 <= lon_max <= 180.0:
            return lon_min, lon_max
        lon_min = (lon_min + 180.0) % 360.0 - 180.0
        lon_max = (lon_max + 180.0) % 360.0 - 180.0
        return lon_min, lon_max

    def _select_region_and_bounds(self, ds: xr.Dataset) -> xr.Dataset:
        """Subset by lat and lon, handling descending lat and dateline crossing."""
        ds = self._maybe_add_bounds(ds)

        # latitude
        if "lat" in ds.coords:
            lat_min, lat_max = self.lat_bounds
            lat_vals = ds["lat"].values
            lat_asc = bool(np.diff(lat_vals).mean() > 0)
            lat_slice = slice(lat_min, lat_max) if lat_asc else slice(lat_max, lat_min)
            ds = ds.sel(lat=lat_slice)

        # longitude
        if "lon" not in ds.coords:
            return ds

        lon_min, lon_max = self.lon_bounds
        if lon_min <= lon_max:
            return ds.sel(lon=slice(lon_min, lon_max))
        else:
            # crosses dateline
            left  = ds.sel(lon=slice(lon_min, 180))
            right = ds.sel(lon=slice(-180, lon_max))
            right = right.assign_coords(lon=self._wrap_to_180(right["lon"]))
            return xr.concat([left, right], dim="lon").sortby("lon")

    def _maybe_add_bounds(self, ds: xr.Dataset) -> xr.Dataset:
        if _HAVE_XCDAT:
            try:
                ds = ds.bounds.add_missing_bounds()
            except Exception:
                pass
        return ds

    def _sync_time_to_reference(self):
        ref_time = self.exp_dict[self.sfc_mask_ref]["time"]
        for k in list(self.exp_dict.keys()):
            if k != self.sfc_mask_ref and "time" in self.exp_dict[k].coords:
                self.exp_dict[k] = self.exp_dict[k].assign_coords(time=ref_time)

    def _debug_print(self, key, path, lats0=None, lons0=None):
        if key != self.debug_key:
            return
        print(f"[DEBUG] {key}")
        if lats0 is not None:
            print("  lats0:", np.asarray(lats0).min(), "→", np.asarray(lats0).max())
        if lons0 is not None:
            print("  lons0:", np.asarray(lons0).min(), "→", np.asarray(lons0).max())
        print("  period:", self.run_catalog[key]["period"])
        print("  path:", path)
        print("  lon(post):", self.exp_dict[key]["lon"].min().item(), "→", self.exp_dict[key]["lon"].max().item())

    def _normalize_period_str(self, s: str) -> str:
        _DASHES = {
            "\u2010": "-",  # hyphen
            "\u2011": "-",  # non-breaking hyphen
            "\u2012": "-",  # figure dash
            "\u2013": "-",  # en dash
            "\u2014": "-",  # em dash
            "\u2212": "-",  # minus sign
        }
        s = unicodedata.normalize("NFKC", s)
        return s.translate(str.maketrans(_DASHES)).strip()
    
    # --- main API ---
    def read(self, debug=True):
        for exp_key, meta in self.run_catalog.items():
            raw_period = self._meta_field(meta, "period")
            period = self._normalize_period_str(raw_period)
            nc_path = os.path.join(self.data_dir, f"{exp_key}.{self.dtype}.{period}.nc")
            if self.verbose:
                print(f"Loading: {nc_path}")
            if not os.path.exists(nc_path):
                raise FileNotFoundError(f"File not found for run '{exp_key}': {nc_path}")

            ds = xcdat_open(nc_path, decode_times=True)

            # pick a reasonable debug variable
            dbg_var = "TS" if "TS" in ds.data_vars else ("FLNS" if "FLNS" in ds.data_vars else None)
            if debug and dbg_var:
                da0 = ds[dbg_var]
                tot0   = int(da0.size)
                valid0 = int(da0.notnull().sum())
                lon_name = "lon" if "lon" in ds.coords else ("longitude" if "longitude" in ds.coords else None)
                if lon_name:
                    print(f"[open] {exp_key}: valid={valid0}/{tot0} ({valid0/tot0:.1%}); "
                          f"lon-range={float(ds[lon_name].min()):.2f}..{float(ds[lon_name].max()):.2f}")
                else:
                    print(f"[open] {exp_key}: valid={valid0}/{tot0} ({valid0/tot0:.1%}); no lon")

            # lon normalize
            ds = self._normalize_longitude(ds)
            if debug and dbg_var:
                daN = ds[dbg_var]
                totN   = int(daN.size)
                validN = int(daN.notnull().sum())
                print(f"[norm] {exp_key}: valid={validN}/{totN} ({validN/totN:.1%}); "
                      f"lon-range={float(ds['lon'].min()):.2f}..{float(ds['lon'].max()):.2f}")

            # region crop (handles descending lat and dateline)
            ds = self._select_region_and_bounds(ds)
            if debug and dbg_var:
                daR = ds[dbg_var]
                totR   = int(daR.size)
                validR = int(daR.notnull().sum())
                print(f"[crop] {exp_key}: valid={validR}/{totR} ({validR/totR:.1%}); "
                      f"extent={self.lon_bounds}×{self.lat_bounds}")

            self.exp_dict[exp_key] = ds
            if debug:
                lats0 = ds["lat"].values if "lat" in ds.coords else None
                lons0 = ds["lon"].values if "lon" in ds.coords else None
                self._debug_print(exp_key, nc_path, lats0, lons0)

        if self.sfc_mask_ref not in self.exp_dict:
            raise KeyError(f"Reference '{self.sfc_mask_ref}' not loaded (after reading).")

        # Align time to reference
        self._sync_time_to_reference()

        # Build land mask using the reference run's period
        ref_meta = self.run_catalog[self.sfc_mask_ref]
        ref_period_raw = self._meta_field(ref_meta, "period")
        ref_period = self._normalize_period_str(ref_period_raw)

        # generate region/land mask consistent with reference
        self.sfc_mask = self.gen_region_mask(
            self.data_dir, self.sfc_mask_ref, self.dtype, ref_period,
            self.sfc_mask_var, self.regnam, debug=debug
        )
        return self.run_catalog, self.exp_dict, self.sfc_mask


In [4]:
class MeanBias2DPlotter:
    """
    Plot 2D fields as (seasons × models) mean bias maps, or as (seasons × decades)
    decadal anomalies (ensemble-aware).

    Modes:
      - plot_type="model_vs_obs": difference vs. a reference dataset in var_dict['ref']
      - plot_type="decadal_anomaly": decadal anomalies vs a baseline decade (per-member baselining, then ensemble collapse)

    Ensemble handling (decadal_anomaly mode):
      - ens_dim: name of ensemble dimension if present (default 'ens')
      - ens_mode: 'mean' (default), 'spread' (stdev), 'qtiles' (median by default)
    """

    def __init__(self, run_dict, df, lmsk, var_dict, seasons, regnam, model_list,
                 plot_type, fgw=35, fgh=25, hs=0.5, ws=0.5,
                 region_catalog=None, constrained_layout=False,
                 # NEW:
                 ref_decade="first", decades=None, ens_dim="ens",
                 ens_mode="mean", ens_quantiles=(0.1, 0.9)):
        self.run_dict   = run_dict
        self.df         = df
        self.lmsk       = lmsk
        self.var_dict   = var_dict
        self.var        = var_dict.get('var')
        if self.var is None:
            raise ValueError("var_dict must include 'var'.")
        self.seasons    = seasons
        self.regnam     = regnam
        self.model_list = model_list
        self.plot_type  = plot_type
        self.fgw        = float(fgw); self.fgh = float(fgh)
        self.hs         = float(hs);  self.ws  = float(ws)
        self.use_constrained = bool(constrained_layout)

        self.nrows = len(seasons); self.ncols = len(model_list)

        self.region_catalog = region_catalog or {}
        self.map_crs, self.trans = self._pick_projection(self.regnam)  # map_crs for axes; trans=PlateCarree
        self.extent = self._region_extent(self.regnam)                 # in PlateCarree coords

        self.cmap, self.clevs = self._build_colormap_levels(self.var_dict)

        # --- NEW options for decadal anomalies / ensembles ---
        self.ref_decade = ref_decade       # 'first'|'last'|int|(start,end)
        self.decades    = decades          # list[int] to keep or None
        self.ens_dim    = ens_dim
        self.ens_mode   = ens_mode         # 'mean'|'spread'|'qtiles'
        self.ens_quantiles = tuple(ens_quantiles)

    # -------------------- tiny utils --------------------
    @staticmethod
    def _meta_get(meta, key, default=None):
        from collections.abc import Mapping
        if isinstance(meta, Mapping):
            return meta.get(key, default)
        return getattr(meta, key, default)

    @staticmethod
    def _coslat_weights(da2d: xr.DataArray, lat_dim="lat", lon_dim="lon"):
        if lat_dim not in da2d.coords:
            raise ValueError(f"Latitude coord '{lat_dim}' not found in {list(da2d.coords.keys())}")
        w = np.cos(np.deg2rad(da2d[lat_dim]))
        return xr.ones_like(da2d) * w

    def _get_spatial_dims(self, da: xr.DataArray):
        dims = list(da.dims)
        lat_dim = next((d for d in dims if "lat" in d.lower()), None)
        lon_dim = next((d for d in dims if "lon" in d.lower()), None)
        if lat_dim and lon_dim: return lat_dim, lon_dim
        if da.ndim >= 2: return dims[-2], dims[-1]
        raise ValueError(f"No suitable spatial dims found in {dims}")

    def _rename_like(self, x, lat_dim, lon_dim):
        rm = {}
        if lat_dim in x.dims: rm[lat_dim] = "lat"
        if lon_dim in x.dims: rm[lon_dim] = "lon"
        return x.rename(rm) if rm else x

    def _as_float(self, x):
        try:
            return float(x)
        except Exception:
            return x.values.item() if hasattr(x, 'values') else float(x)

    def _has_ens(self, da: xr.DataArray) -> bool:
        return (self.ens_dim in da.dims)

    def _broadcast_weights(self, w: xr.DataArray, da: xr.DataArray) -> xr.DataArray:
        # ensure w(time) can broadcast to da(..., time, lat, lon[, ens])
        for d in da.dims:
            if d not in w.dims and d != "time":
                w = w.expand_dims({d: da.coords[d]})
        return w

    # ---------- mask application with alignment + diagnostics ----------
    def _apply_mask(self, da: xr.DataArray, debug: bool = False) -> xr.DataArray:
        """Align self.lmsk (if DataArray) to da's horizontal grid and apply it."""
        if getattr(self, "lmsk", None) is None:
            return da

        before_valid = int(da.notnull().sum())

        # pick a 2D target grid to align the mask
        tgt2d = da.isel(time=0, drop=True) if "time" in da.dims else da

        if isinstance(self.lmsk, xr.DataArray):
            m = self.lmsk
            lat_dim, lon_dim = self._get_spatial_dims(m)
            m = self._rename_like(m, lat_dim, lon_dim)
            extra = [d for d in m.dims if d not in ("lat", "lon")]
            if extra:
                m = m.isel({d: 0 for d in extra})
            if (("lat" in m.coords) and ("lon" in m.coords)
                and (not m.lat.identical(tgt2d.lat) or not m.lon.identical(tgt2d.lon))):
                # nearest is fine for binary masks
                m = m.interp_like(tgt2d, method="nearest")
            mask_bool = (m > 0).astype(bool)
        else:
            # numpy/list mask → wrap to target grid shape
            mask_bool = xr.DataArray(np.asarray(self.lmsk, dtype=bool),
                                     coords=tgt2d.coords, dims=tgt2d.dims)

        # broadcast mask over time/ens if present
        da = da.where(mask_bool)

        after_valid = int(da.notnull().sum())
        if debug:
            masked_out = before_valid - after_valid
            frac = masked_out / before_valid if before_valid > 0 else np.nan
            print(f"[mask-apply] {self.var}: before={before_valid}, after={after_valid}, "
                  f"masked={masked_out} ({frac:.1%})")
            if before_valid > 0 and after_valid < max(100, int(0.001*before_valid)):
                print("[mask-apply] WARNING: ~all data removed — check mask/grid alignment.")
        return da

    # ---------- seasonal extraction (day-weighted) ----------
    def _extract_seasonal_mean(
        self,
        model_key: str,
        var: Optional[str] = None,
        vunt: Optional[str] = None,
        vfac: float = 1.0,
        debug: bool = True,
    ):
        vname = var or self.var
        ds = self.df[model_key]
        da = ds[vname].astype("float64") * float(vfac)

        if vunt is not None:
            da.attrs["units"] = vunt

        # Apply land/region mask (aligned)
        da = self._apply_mask(da, debug=debug)

        # day-weighted seasonal mean
        w = da["time"].dt.days_in_month.astype("float64")
        w_eff = xr.where(da.isnull(), 0.0, w)

        num = (da * w_eff).groupby("time.season").sum("time", skipna=True)
        den = w_eff.groupby("time.season").sum("time", skipna=True)
        seas_mean = num / den

        out: dict[str, xr.DataArray] = {}
        tmpl_src = da.isel(time=0, drop=True) if da.sizes.get("time", 0) > 0 else da.isel(..., drop=True)
        tmpl = tmpl_src.copy(deep=True) * np.nan
        tmpl.attrs["units"] = vunt if vunt is not None else da.attrs.get("units", "")

        for sea in self.seasons:
            if sea == "ANN":
                num_ann = (da * w_eff).sum("time", skipna=True)
                den_ann = w_eff.sum("time", skipna=True)
                out[sea] = num_ann / den_ann
            else:
                if "season" in seas_mean.coords and sea in seas_mean["season"].values:
                    out[sea] = seas_mean.sel(season=sea)
                else:
                    out[sea] = tmpl
            if vunt is not None:
                out[sea].attrs["units"] = vunt

        return out

    # ---------- metrics (model_vs_obs mode) ----------
    def metrics_rmse_pcc(self, aa, bb, w, lat_dim, lon_dim):
        aa = self._rename_like(aa, lat_dim, lon_dim)
        bb = self._rename_like(bb, lat_dim, lon_dim)
        if hasattr(w, "dims"): w = self._rename_like(w, lat_dim, lon_dim)

        aa, bb = xr.align(aa, bb, join="inner")
        if hasattr(w, "dims") and set(["lat","lon"]).issubset(w.dims):
            w = w.sel(lat=aa["lat"], lon=aa["lon"])

        mask = aa.notnull() & bb.notnull()
        aa = aa.where(mask); bb = bb.where(mask)
        if hasattr(w, "dims") and set(w.dims) == set(["lat","lon"]):
            w = w.where(mask)

        if (np.isnan(aa).all() or np.isnan(bb).all() or
            (hasattr(w, "dims") and np.isnan(w).all())):
            scalar_nan = xr.full_like(aa.isel(lat=0, lon=0), np.nan)
            return scalar_nan, scalar_nan

        dims = ["lat", "lon"]
        rmse = xs.rmse(bb, aa, dim=dims, weights=w, skipna=True)
        pcor = xs.pearson_r(bb, aa, dim=dims, weights=w, skipna=True)
        return rmse, pcor

    def _derive_hor_metrics_data(self, mod_dict, obs_dict=None, var=None, debug=False):
        vname = var or self.var
        dat_dict = {}; mec_dict = {}
        for sea in self.seasons:
            mod_da = mod_dict[sea]
            lat_dim, lon_dim = self._get_spatial_dims(mod_da)
            weights = self._coslat_weights(mod_da, lat_dim=lat_dim, lon_dim=lon_dim)

            mec = {}
            try:
                nan_val = xr.full_like(mod_da.isel({lat_dim: 0, lon_dim: 0}), np.nan)
            except Exception:
                nan_val = xr.DataArray(np.nan)

            # mask invalid model values
            mask_mod = mod_da.notnull()
            mod_da = mod_da.where(mask_mod)
            weights = weights.where(mask_mod)

            if int(weights.notnull().sum()) == 0:
                mec["mean"] = nan_val; mec["rmsm"] = nan_val
                mec["rmse"] = nan_val; mec["pcor"] = nan_val
                dat_dict[sea] = mod_da; mec_dict[sea] = mec
                continue

            # weighted mean and RMS magnitude
            wsum = weights.sum((lat_dim, lon_dim), skipna=True)
            mean_mag = (abs(mod_da) * weights).sum((lat_dim, lon_dim), skipna=True) / wsum
            rmsm = np.sqrt(((mod_da ** 2) * weights).sum((lat_dim, lon_dim), skipna=True) / wsum)
            mec["mean"] = mean_mag; mec["rmsm"] = rmsm

            if obs_dict is not None:
                obs_da = obs_dict[sea]
                mask = mod_da.notnull() & obs_da.notnull()
                aa = mod_da.where(mask); bb = obs_da.where(mask)
                w_common = weights.where(mask)

                if int(mask.sum()) == 0:
                    rmse = nan_val; pcor = nan_val
                else:
                    rmse, pcor = self.metrics_rmse_pcc(aa, bb, w_common, lat_dim, lon_dim)

                mec["rmse"] = rmse; mec["pcor"] = pcor
                if debug:
                    for lab, val in (("RMSE", rmse), ("PCC", pcor)):
                        try: print(lab + ":", float(val))
                        except Exception: print(lab + ":", val)

                dat = (mod_da - obs_da)
            else:
                dat = mod_da

            dat_dict[sea] = dat; mec_dict[sea] = mec
        return dat_dict, mec_dict

    # ---------- region/projection ----------
    def _pick_projection(self, regnam):
        data_crs = ccrs.PlateCarree()
        if regnam == 'Antarctic':
            map_crs = ccrs.SouthPolarStereo(central_longitude=0, true_scale_latitude=-71)
        elif regnam == 'PolarN':
            map_crs = ccrs.NorthPolarStereo(central_longitude=0, true_scale_latitude=71)
        elif regnam == 'CONUS':
            map_crs = ccrs.LambertConformal(
                central_longitude=-96.0, central_latitude=39.0, standard_parallels=(33, 45)
            )
        elif regnam == "Atlantic":
            map_crs = data_crs  # plain lat-lon for Atlantic
        else:
            map_crs = data_crs
        return map_crs, data_crs

    def _region_extent(self, regnam):
        lon_min, lon_max, lat_min, lat_max = -180.0, 180.0, -90.0, 90.0
        if self.region_catalog and regnam in self.region_catalog:
            region = self.region_catalog[regnam]
            if hasattr(region, "lat_range") and hasattr(region, "lon_range"):
                (lat_min, lat_max) = region.lat_range
                (lon_min, lon_max) = region.lon_range
            elif isinstance(region, dict) and "lat_range" in region and "lon_range" in region:
                (lat_min, lat_max) = region["lat_range"]
                (lon_min, lon_max) = region["lon_range"]
            else:
                try:
                    (lat_min, lat_max), (lon_min, lon_max) = region
                except Exception:
                    pass

        if regnam == 'Atlantic':
            lon_min, lon_max = -80.0, 5.0
            lat_min, lat_max =  5.0, 80.0
        elif regnam == 'CONUS':
            lon_min, lon_max = -130.0, -65.0
            lat_min, lat_max = 25.0, 55.0
        elif regnam == 'Antarctic':
            lat_min, lat_max = -90.0, -50.0
        elif regnam == 'PolarN':
            lat_min, lat_max = 50.0, 90.0

        lon_min = (lon_min + 180.0) % 360.0 - 180.0
        lon_max = (lon_max + 180.0) % 360.0 - 180.0
        return [lon_min, lon_max, lat_min, lat_max]

    # ---------- visuals ----------
    @staticmethod
    def _build_colormap_levels(var_dict):
        color_list4 = ['darkblue', 'lightblue', 'white', 'lightyellow', 'orange', 'firebrick']
        nodes = [0.0, 0.3, 0.5, 0.500001, 0.8, 1.0]  # strictly increasing (matplotlib >= 3.11)
        cmap = matcolors.LinearSegmentedColormap.from_list('mybluewhitered', list(zip(nodes, color_list4)))
        vmin = var_dict['min']; vmax = var_dict['max']; nlev = var_dict['nlev']
        clevs = np.linspace(vmin, vmax, nlev)
        return cmap, clevs

    def _add_lat_lon_labels(self, axes, k, fontz, r=0.975):
        ax = axes.flat[k]
        lons = np.arange(-180, 181, 30)
        labs = ['180','150W','120W','90W','60W','30W','0   ','30E','60E','90E','120E','150E','180']
        thetas = np.deg2rad(90 - lons)
        for lon, lab, th in zip(lons, labs, thetas):
            x = 0.5 + r * np.cos(th)
            y = 0.5 + r * np.sin(th)
            ax.text(x, y, lab, fontsize=fontz*0.9, ha='center', va='center',
                    transform=ax.transAxes, zorder=6, clip_on=False)

    # ---------- NEW: seasonal yearly means (ensemble-safe) ----------
    def _seasonal_yearly_means(self, da: xr.DataArray):
        """Return dict: season -> DataArray([ens,] year, lat, lon). Day-weighted.
           DJF assigned to the January year (Dec shifted +1)."""
        da = self._apply_mask(da, debug=False)

        w = da["time"].dt.days_in_month.astype("float64")
        w = self._broadcast_weights(w, da)
        w_eff = xr.where(da.isnull(), 0.0, w)

        year  = da["time"].dt.year
        month = da["time"].dt.month
        djf_year = xr.where(month == 12, year + 1, year)

        out = {}

        # ANN
        num = (da * w_eff).groupby(year).sum("time", skipna=True)
        den = w_eff.groupby(year).sum("time", skipna=True)
        ann = (num / den).rename({"time": "_drop"})
        ann = ann.rename({list(ann.dims)[list(ann.dims).index(year.name)]: "year"})
        ann = ann.assign_coords(year=np.unique(year))
        if "ANN" in self.seasons:
            out["ANN"] = ann

        # DJF (Jan-year)
        if "DJF" in self.seasons:
            msk = month.isin([12, 1, 2])
            num = (da.where(msk) * w_eff.where(msk)).groupby(djf_year).sum("time", skipna=True)
            den = w_eff.where(msk).groupby(djf_year).sum("time", skipna=True)
            djf = (num / den).rename({"time": "_drop"})
            djf = djf.rename({list(djf.dims)[list(djf.dims).index(djf_year.name)]: "year"})
            djf = djf.assign_coords(year=np.unique(djf_year))
            out["DJF"] = djf

        for sea, months in (("MAM",[3,4,5]), ("JJA",[6,7,8]), ("SON",[9,10,11])):
            if sea in self.seasons:
                msk = month.isin(months)
                num = (da.where(msk) * w_eff.where(msk)).groupby(year).sum("time", skipna=True)
                den = w_eff.where(msk).groupby(year).sum("time", skipna=True)
                smean = (num / den).rename({"time": "_drop"})
                smean = smean.rename({list(smean.dims)[list(smean.dims).index(year.name)]: "year"})
                smean = smean.assign_coords(year=np.unique(year))
                out[sea] = smean

        return out

    # ---------- NEW: decade helpers / ensemble collapse ----------
    @staticmethod
    def _to_decade_index(years: np.ndarray) -> np.ndarray:
        years = np.asarray(years, dtype=int)
        return (years // 10) * 10

    def _decadal_means(self, yrly: xr.DataArray):
        """Average yearly seasonal means into decadal means (equal-year weights), keep ensemble dim if present."""
        if "year" not in yrly.dims:
            raise ValueError("Input must have a 'year' dimension.")
        decades = self._to_decade_index(yrly["year"].values)
        yrly = yrly.assign_coords(decade=("year", decades))
        return yrly.groupby("decade").mean("year", skipna=True)

    def _pick_baseline_decade(self, decades_avail: np.ndarray):
        if isinstance(self.ref_decade, (tuple, list)):
            d0, d1 = int(self.ref_decade[0]), int(self.ref_decade[1])
            cand = [d for d in decades_avail if d0 <= d <= d1]
            return cand[0] if cand else int(decades_avail.min())
        if isinstance(self.ref_decade, str):
            return int(decades_avail.min()) if self.ref_decade.lower()=="first" else int(decades_avail.max())
        return int(self.ref_decade)

    def _collapse_ens(self, x: xr.DataArray):
        if not self._has_ens(x):
            return x
        if self.ens_mode == "mean":
            return x.mean(self.ens_dim, skipna=True)
        if self.ens_mode == "spread":
            return x.std(self.ens_dim, skipna=True, ddof=1 if x.sizes[self.ens_dim] > 1 else 0)
        if self.ens_mode == "qtiles":
            # median by default; change to self.ens_quantiles[0]/[1] if preferred
            return x.quantile(0.5, dim=self.ens_dim, skipna=True)
        raise NotImplementedError(f"ens_mode='{self.ens_mode}' is not supported here.")

    def _compute_decadal_anomaly(self, ds: xr.Dataset, vname: str):
        """
        Return dict[season] -> DataArray(decade, lat, lon) after ensemble collapse.
        Steps: per-member yearly seasonal means → per-member decadal means → per-member anomaly → collapse.
        """
        da = ds[vname].astype("float64")
        seas_yearly = self._seasonal_yearly_means(da)

        out = {}
        for sea in self.seasons:
            yrly = seas_yearly.get(sea)
            if yrly is None or yrly.size == 0:
                continue

            dmean = self._decadal_means(yrly)   # dims: (decade, [ens,] lat, lon)
            decs = dmean["decade"].values

            if self.decades is not None:
                keep = np.intersect1d(np.asarray(self.decades, int), decs)
                dmean = dmean.sel(decade=keep)
                decs = keep
            if len(decs) == 0:
                out[sea] = dmean
                continue

            base_dec = self._pick_baseline_decade(decs)
            base = dmean.sel(decade=base_dec)

            # anomaly per member
            anom = dmean - base

            # collapse over ensemble if present
            anom_c = self._collapse_ens(anom)

            # order (decade, lat, lon, ...)
            for nm in ("lat","lon"):
                if nm in anom_c.dims:
                    anom_c = anom_c.transpose(*(d for d in anom_c.dims if d in ("decade","lat","lon")), ...)
            anom_c.attrs["units"] = ds[vname].attrs.get("units", "")
            anom_c.attrs["long_name"] = f"{vname} {sea} decadal anomaly (baseline {base_dec}s; ens={self.ens_mode})"
            out[sea] = anom_c
        return out

    # ---------- main ----------
    def plot(self, fontz=12, savepath=None, show=False, return_fig=False, return_artists=False, debug=False):
        # --- Special branch: decadal anomalies (seasons × decades, one model) ---
        if self.plot_type == "decadal_anomaly":
            if len(self.model_list) != 1:
                print("[warn] decadal_anomaly currently plots one model; using the first in model_list.")
            model = self.model_list[0]
            ds = self.df[model]
            vname = self.var; vunt = self.var_dict['unit']; vfac = self.var_dict['fac']

            ds = ds.copy()
            ds[vname] = ds[vname].astype("float64") * float(vfac)
            if vunt: ds[vname].attrs["units"] = vunt

            # compute decadal anomalies
            anom_dict = self._compute_decadal_anomaly(ds, vname)  # season -> (decade, lat, lon)

            # build decades axis from first available season
            first_sea = next((s for s in self.seasons if (s in anom_dict and anom_dict[s].sizes.get("decade", 0) > 0)), None)
            if first_sea is None:
                raise ValueError("No seasonal decadal anomalies available to plot.")
            decades = list(map(int, anom_dict[first_sea]["decade"].values))
            if len(decades) == 0:
                raise ValueError("No decades available after filtering.")

            # create figure (seasons × decades)
            nrows = len(self.seasons); ncols = len(decades)
            fig, axes = plt.subplots(
                nrows=nrows, ncols=ncols, figsize=(self.fgw, self.fgh),
                subplot_kw={'projection': self.map_crs},
                constrained_layout=self.use_constrained,
            )
            axes = np.atleast_2d(axes)

            fontz = fontz * self.fgw / max(self.fgh, 1.0)
            clevs = self.clevs; cmap = self.cmap
            norm = matcolors.BoundaryNorm(clevs, cmap.N)

            if self.ens_mode == "spread":
                cbar_title = f"{vname} anomaly stdev ({vunt})"
            elif self.ens_mode == "qtiles":
                cbar_title = f"{vname} anomaly median ({vunt})"
            else:
                cbar_title = f"{vname} anomaly ({vunt})"

            lon0, lon1, lat0, lat1 = self.extent
            eps = 1e-6
            if lon0 == lon1: lon0 -= eps; lon1 += eps
            if lat0 == lat1: lat0 -= eps; lat1 += eps
            safe_extent = [lon0, lon1, lat0, lat1]

            contour = np.empty((nrows, ncols), dtype=object)

            for i, sea in enumerate(self.seasons):
                A = anom_dict.get(sea)
                for j, dec in enumerate(decades):
                    ax = axes[i, j]
                    ax.set_extent(safe_extent, crs=self.trans)

                    if isinstance(ax.projection, (ccrs.NorthPolarStereo, ccrs.SouthPolarStereo)):
                        theta = np.linspace(0, 2*np.pi, 256)
                        circle = matpath(0.5 + 0.5*np.vstack([np.sin(theta), np.cos(theta)]).T)
                        ax.set_boundary(circle, transform=ax.transAxes)

                    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, zorder=3)
                    ax.add_feature(cfeature.LAKES, linewidth=0.5, edgecolor='black', facecolor='None', zorder=4)

                    gl = ax.gridlines(draw_labels=False, linewidth=0.6, linestyle="--", color="gray", zorder=5)
                    # (reuse region-specific tick logic if desired)

                    panel = None if (A is None or A.size == 0) else A.sel(decade=dec).squeeze()
                    if (panel is not None) and panel.notnull().any():
                        contour[i, j] = panel.plot.contourf(
                            ax=ax, transform=self.trans,
                            levels=clevs, cmap=cmap, norm=norm,
                            add_colorbar=False, extend='both'
                        )
                    else:
                        sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
                        contour[i, j] = sm
                        ax.text(0.5, 0.5, "No data", transform=ax.transAxes,
                                ha="center", va="center", fontsize=max(fontz*0.9, 9),
                                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.5", alpha=0.8))

                    ax.set_title(f"{sea} — {dec}s", fontsize=fontz*0.9, y=1.06)

            if not self.use_constrained:
                plt.subplots_adjust(hspace=self.hs, wspace=self.ws)

            # shared colorbar per row
            for i in range(nrows):
                im = contour[i, ncols - 1]
                cbar = fig.colorbar(im, ax=axes[i, :], orientation='vertical',
                                    fraction=0.20, pad=0.02, shrink=0.80)
                cbar.set_label(cbar_title, rotation=270, labelpad=20, fontsize=fontz*0.9)
                cbar.ax.tick_params(labelsize=max(fontz*0.85, 8))

            # default savepath
            if savepath is None:
                last_meta = self.run_dict[self.model_list[0]]
                period = self._meta_get(last_meta, 'period', 'NA')
                savepath = f"fig_2d_decadal_anom_{vname}_{self.regnam}_{period}.pdf"

            if return_fig:
                return (fig, axes) if not return_artists else (fig, axes, {'contour': contour})

            plt.ioff()
            fig.savefig(savepath, bbox_inches='tight')
            if show: plt.show()
            plt.close(fig)
            return savepath

        # --- Original branch: model_vs_obs (seasons × models) ---
        nrows, ncols = self.nrows, self.ncols
        fig, axes = plt.subplots(
            nrows=nrows, ncols=ncols, figsize=(self.fgw, self.fgh),
            subplot_kw={'projection': self.map_crs},
            constrained_layout=self.use_constrained,
        )

        fontz = fontz * self.fgw / max(self.fgh, 1.0)
        vname = self.var; vunt = self.var_dict['unit']; vfac = self.var_dict['fac']
        refmodel = self.var_dict.get('ref', None)
        clevs = self.clevs; cmap = self.cmap
        norm = matcolors.BoundaryNorm(clevs, cmap.N)
        cbar_title = f'{vname} ({vunt})'

        seas_cache = {m: self._extract_seasonal_mean(m, vname, vunt, vfac, debug=debug)
                      for m in self.model_list}
        obs_cache = (self._extract_seasonal_mean(refmodel, vname, vunt, vfac, debug=debug)
                     if (self.plot_type == "model_vs_obs" and refmodel) else None)

        contour = np.empty((nrows, ncols), dtype=object)

        lon0, lon1, lat0, lat1 = self.extent
        eps = 1e-6
        if lon0 == lon1: lon0 -= eps; lon1 += eps
        if lat0 == lat1: lat0 -= eps; lat1 += eps
        safe_extent = [lon0, lon1, lat0, lat1]

        for j, model in enumerate(self.model_list):
            mod_dict = seas_cache[model]
            if self.plot_type == "model_vs_obs" and obs_cache is not None:
                dat_dict, mec_dict = self._derive_hor_metrics_data(mod_dict, obs_cache, var=vname, debug=debug)
            else:
                dat_dict, mec_dict = self._derive_hor_metrics_data(mod_dict, obs_dict=None, var=vname, debug=debug)

            for i, sea in enumerate(self.seasons):
                ax = axes.flat[i * ncols + j]
                ax.set_extent(safe_extent, crs=self.trans)

                # Only clip to circular boundary on polar projections
                if isinstance(ax.projection, (ccrs.NorthPolarStereo, ccrs.SouthPolarStereo)):
                    theta = np.linspace(0, 2*np.pi, 256)
                    circle = matpath(0.5 + 0.5*np.vstack([np.sin(theta), np.cos(theta)]).T)
                    ax.set_boundary(circle, transform=ax.transAxes)

                ax.add_feature(cfeature.COASTLINE, linewidth=0.5, zorder=3)
                ax.add_feature(cfeature.LAKES, linewidth=0.5, edgecolor='black', facecolor='None', zorder=4)

                # gridlines
                gl = ax.gridlines(
                    draw_labels=True,
                    linewidth=0.6, linestyle="--", color="gray",
                    x_inline=False, y_inline=False, zorder=5
                )
                if self.regnam == 'Antarctic':
                    gl.xlocator = mticker.FixedLocator(np.arange(lon0, lon1, 30))
                    gl.ylocator = mticker.FixedLocator(np.arange(lat0, lat1, 10))
                elif self.regnam == 'PolarN':
                    gl.xlocator = mticker.FixedLocator(np.arange(lon0, lon1, 30))
                    gl.ylocator = mticker.FixedLocator(np.arange(lat0, lat1, 10))
                elif self.regnam == 'Atlantic':
                    gl.xlocator = mticker.FixedLocator(np.arange(lon0, lon1, 20))
                    gl.ylocator = mticker.FixedLocator(np.arange(lat0, lat1, 20))
                    xticks = np.arange(lon0, lon1, 20)
                    yticks = np.arange(lat0, lat1, 20)
                    ax.set_xticks(xticks, crs=ccrs.PlateCarree())
                    ax.set_yticks(yticks, crs=ccrs.PlateCarree())
                    ax.set_yticklabels([])
                    ax.set_xticklabels([])
                else:
                    dlon = 30 if (lon1 - lon0) > 90 else 10
                    dlat = 30 if (lat1 - lat0) > 90 else 10
                    gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, dlon))
                    gl.ylocator = mticker.FixedLocator(np.arange(-90, 91, dlat))

                # remove all labels first
                gl.top_labels = False
                gl.right_labels = False
                gl.left_labels = False
                gl.bottom_labels = False

                # add only for leftmost column and bottom row
                if j == 0:      # leftmost column
                    gl.left_labels = True
                if i == self.nrows-1:   # bottom row
                    gl.bottom_labels = True

                # style
                gl.xlabel_style = {"size": max(fontz*0.8, 8)}
                gl.ylabel_style = {"size": max(fontz*0.8, 8)}
                ax.tick_params(labelsize=max(fontz*0.8, 8))

                # field
                vplt = dat_dict[sea].squeeze()
                if debug:
                    valid = int(vplt.notnull().sum())
                    total = int(vplt.size)
                    missing = total - valid
                    frac = valid / total if total > 0 else np.nan
                    vmin = float(vplt.min(skipna=True)) if valid else np.nan
                    vmax = float(vplt.max(skipna=True)) if valid else np.nan
                    print(f"[diag] {model}-{sea}: shape={tuple(vplt.shape)}, "
                          f"min={vmin:.3f}, max={vmax:.3f}, valid={valid}/{total} ({frac:.1%}), missing={missing}")

                if vplt.notnull().any():
                    if self.regnam == 'Atlantic':
                        contour[i, j] = ax.contourf(
                            vplt["lon"].values, vplt["lat"].values, vplt.values,
                            levels=self.clevs, cmap=self.cmap, norm=norm, transform=self.trans, extend='both'
                        )
                    else:
                        contour[i, j] = vplt.plot.contourf(
                            ax=ax, transform=self.trans,
                            levels=self.clevs, cmap=self.cmap, norm=norm, add_colorbar=False, extend='both'
                        )
                else:
                    sm = ScalarMappable(norm=norm, cmap=self.cmap); sm.set_array([])
                    contour[i, j] = sm
                    ax.text(0.5, 0.5, "No data", transform=ax.transAxes,
                            ha="center", va="center", fontsize=max(fontz*0.9, 9),
                            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.5", alpha=0.8))

                # titles/metrics
                model_meta = self.run_dict[model]
                model_name = self._meta_get(model_meta, 'name', str(model))
                ax.set_title(f"{model_name} ({sea})", fontsize=fontz*0.9, y=1.06)

                if self.plot_type == "model_vs_obs" and ('rmse' in mec_dict[sea] and 'pcor' in mec_dict[sea]):
                    rmse_val = self._as_float(mec_dict[sea]['rmse'])
                    pcor_val = self._as_float(mec_dict[sea]['pcor'])
                    label_str = f"RMSE={rmse_val:.2f}\nPCC={pcor_val:.2f}"
                else:
                    mean_val = mec_dict[sea]['mean']
                    mean_val = mean_val.values.item() if hasattr(mean_val, 'values') else float(mean_val)
                    label_str = f"Mean={mean_val:.2f}"

                at = AnchoredText(
                    label_str,
                    loc='lower right',
                    prop=dict(size=max(fontz*0.75, 8)),
                    frameon=True,
                    borderpad=0.1,
                )
                at.patch.set_boxstyle('round,pad=0.1')
                at.patch.set_facecolor('white')
                at.patch.set_alpha(0.65)
                at.patch.set_edgecolor('0.4')
                ax.add_artist(at)

        if not self.use_constrained:
            plt.subplots_adjust(hspace=self.hs, wspace=self.ws)

        # shared colorbar per row
        for i in range(self.nrows):
            im = contour[i, self.ncols - 1]
            cbar = fig.colorbar(im, ax=axes[i, :], orientation='vertical',
                                fraction=0.20, pad=0.02, shrink=0.80)
            cbar.set_label(f'{vname} ({vunt})',
               rotation=270, labelpad=20, fontsize=fontz*0.9)
            cbar.ax.tick_params(labelsize=max(fontz*0.85, 8))

        # default savepath
        if savepath is None:
            last_model = self.model_list[-1]
            last_meta = self.run_dict[last_model]
            period = self._meta_get(last_meta, 'period', 'NA')
            ref_str = self.var_dict.get('ref', 'NA')
            savepath = f"fig_2d_bias_{self.plot_type}_{ref_str}_{vname}_{self.regnam}_{period}.pdf"

        if return_fig:
            return (fig, axes) if not return_artists else (fig, axes, {'contour': contour})

        plt.ioff()
        fig.savefig(savepath, bbox_inches='tight')
        if show:
            plt.show()
        plt.close(fig)
        return savepath


In [5]:
if __name__ == "__main__":
    # --- region settings (config/regions.json) ---
    from regions import region_catalogs, region_settings

    RS = region_settings(REGION)
    RUN_CATALOG, VARIABLE_CATALOG = region_catalogs(REGION)

    # --- paths ---
    data_path = CLIMO_DIR
    out_path  = DIAG_DIR
    fig_path  = FIG_DIR

    # ===========================
    # Controls
    # ===========================
    regnam       = REGION
    dtype        = "climo"

    # Choose one: "decadal_anomaly" (no obs needed) or "model_vs_obs"
    plot_type    = "decadal_anomaly"   # <- change to "model_vs_obs" if needed

    seasons      = ["ANN", "DJF", "JJA"]

    # For model_vs_obs only
    refexp         = RS["refexp"]
    sfc_mask_ref   = RS["sfc_mask_ref"]
    sfc_mask_kind  = RS["sfc_mask_kind"]
    sfc_mask_var   = RS["sfc_mask_var"]

    # Decadal-anomaly options
    ref_decade     = "first"                 # "first" | "last" | int year (e.g., 1850) | (start, end) tuple
    decades_keep   = None                    # e.g., [1850, 1860, ..., 2010] or None for all available
    ens_dim        = "ens"
    ens_mode       = "mean"                  # "mean" | "spread" | "qtiles"

    # --- echo metadata safely (refexp used only to print and for model_vs_obs) ---
    run_info = RUN_CATALOG[refexp]
    try:
        ref_name  = run_info.name
        ref_period = run_info.period
    except AttributeError:
        ref_name  = run_info["name"]
        ref_period = run_info["period"]

    reg_info = REGION_CATALOG[regnam]
    try:
        (lat_min, lat_max) = reg_info.lat_range
        (lon_min, lon_max) = reg_info.lon_range
    except AttributeError:
        (lat_min, lat_max), (lon_min, lon_max) = reg_info

    print(f"reference data and period: {ref_name}, {ref_period}")
    print(f"processed region: {regnam}, lat=({lat_min},{lat_max}), lon=({lon_min},{lon_max})")

    out_dir = f"{out_path}/{regnam}"
    fig_dir = f"{fig_path}/{regnam}"
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(fig_dir, exist_ok=True)

    # --- read regionalized datasets ---
    reader = ExpDataReader(
        data_dir=data_path,
        regnam=regnam,
        dtype=dtype,
        sfc_mask_ref=sfc_mask_ref,
        sfc_mask_var=sfc_mask_var,
        sfc_mask_kind=sfc_mask_kind,
        verbose=True,
    )
    run_catalog, exp_dict, surface_mask = reader.read()

    print("Loaded runs:")
    pprint.pprint(list(run_catalog.keys()))

    # models to plot (exclude the reference/mask datasets for model_vs_obs)
    all_runs = list(run_catalog.keys())
    base_exclusions = {
        'MODIS.observation', "HadISST.analysis", 'CERES_EBAF.observation',
        'ERA5.ensmn', 'MERRA2.analysis', 'ERA5.analysis',
        'v3.LR.amip_ensmn', '20250327.v3.SORRME3r3.piControl.alfred2.chrysalis',
    }

    if plot_type == "model_vs_obs":
        model_list = [k for k in all_runs if k not in base_exclusions | {refexp, sfc_mask_ref}]
    else:
        # decadal anomaly: you can plot any single model per figure
        # pick a default (first non-obs run), or set explicitly below
        candidates = [k for k in all_runs if k not in base_exclusions]
        model_list = [candidates[0]] if candidates else [all_runs[0]]
        print(f"[decadal_anomaly] Using model: {model_list[0]}")

    # --- figure scaling (rough) ---
    nseas = max(len(seasons), 1)
    ncols = max(len(model_list), 1) if plot_type == "model_vs_obs" else 6  # guess ~6 decades; class will adapt
    base_w, base_h = 28.0, 24.0
    fcol = ncols / 5.0
    frow = nseas / 8.0
    fgw  = base_w * fcol
    fgh  = base_h * frow
    ws, hs = 0.1, 0.05
    fontz = 6

    # helper to sanitize periods used in filenames
    save_period = ref_period.translate({
        ord("\u2010"): ord("-"),
        ord("\u2011"): ord("-"),
        ord("\u2012"): ord("-"),
        ord("\u2013"): ord("-"),
        ord("\u2014"): ord("-"),
        ord("\u2212"): ord("-"),
    })

    # --- loop over variables ---
    for var_key in list(VARIABLE_CATALOG.keys()):
        vinfo = VARIABLE_CATALOG[var_key]
        # handle both dict and object styles
        if isinstance(vinfo, dict):
            v_unit = vinfo.get("unit", "")
            v_min  = vinfo.get("min", None)
            v_max  = vinfo.get("max", None)
            v_nlev = vinfo.get("nlev", 11)
            v_ref  = vinfo.get("ref", None)
            v_fac  = vinfo.get("fac", 1.0)
        else:
            v_unit = getattr(vinfo, "unit", "")
            v_min  = getattr(vinfo, "min", None)
            v_max  = getattr(vinfo, "max", None)
            v_nlev = getattr(vinfo, "nlev", 11)
            v_ref  = getattr(vinfo, "ref", None)
            v_fac  = getattr(vinfo, "fac", 1.0)

        var_dict = {
            "var":  var_key,
            "unit": v_unit,
            "min":  v_min,
            "max":  v_max,
            "nlev": v_nlev,
            "ref":  v_ref,   # not used in decadal_anomaly
            "fac":  v_fac,
        }

        # --- availability checks ---
        if plot_type == "model_vs_obs":
            # need variable in ref + each model
            needed = [refexp] + model_list
            missing = [k for k in needed if var_key not in exp_dict[k].data_vars]
            if missing:
                print(f"[skip] {var_key}: missing in {missing}")
                continue
        else:
            # decadal anomaly: need variable only in the chosen model
            need = model_list[0]
            if var_key not in exp_dict[need].data_vars:
                print(f"[skip] {var_key}: missing in '{need}'")
                continue

        # --- init plotter ---
        plotter = MeanBias2DPlotter(
            run_dict=run_catalog,
            df=exp_dict,
            lmsk=surface_mask,            # already aligned by reader
            var_dict=var_dict,
            seasons=seasons,
            regnam=regnam,
            model_list=model_list,
            plot_type=plot_type,
            fgw=fgw, fgh=fgh,
            ws=ws, hs=hs,
            region_catalog=REGION_CATALOG,
            # decadal/ensemble knobs (safe to pass; ignored in model_vs_obs)
            ref_decade=ref_decade,
            decades=decades_keep,
            ens_dim=ens_dim,
            ens_mode=ens_mode,
        )

        # --- build output filename & save directly from class ---
        if plot_type == "model_vs_obs":
            outname = f"fig_mean_bias_{plot_type}_{refexp}_{var_key}_{regnam}_{save_period}.pdf"
        else:
            target_model = model_list[0]
            # the class labels baseline decade in the panel titles; period here is from refexp, but that’s okay.
            outname = f"fig_decadal_anom_{target_model}_{var_key}_{regnam}_{save_period}_{ens_mode}.pdf"

        desired_savepath = os.path.join(fig_dir, outname)

        try:
            # pass savepath so we don't need to move afterwards
            produced = plotter.plot(show=False, fontz=fontz, savepath=desired_savepath)
            print(f"[ok] saved: {produced}")
        except Exception as e:
            print(f"[fail] {var_key}: {e}")


KeyError: 'Arctic'